# DMRGT Phase 3 — single committed session (§0 of the rework)

One `Save Version → Save & Run All (Commit)` run: **grid (140 runs) →
val-only winner selection → in-session frozen graph → `dmrgt_frozen`
(20 runs) → significance tests → output verification**. No `RUN_GRID`
branch, no mid-run downloads, no dataset version bumps, no restarts.

`dmrgt_all3` is trained ONCE per grid variant (§1): each run emits val
AND test rows, the winner is picked from val rows only, and the
winner's already-existing test rows become the final dynamic-arm
result — read for the first time in Cell 6, after selection is locked.
160 runs total, not 180.

Settings that must hold before committing: **GPU T4 x2** (never P100 —
sm_60 has no kernels in this torch build), **Internet OFF**, both
datasets attached (`dmrgt-phase-3-cache`, `dmrgt-phase-3-code`) and the
session RESTARTED after attaching. Cell 0's code fingerprint must equal
the one printed by `kaggle/build_code_dataset_phase3.sh`.

*Research project for academic purposes only — not investment advice.*

In [ ]:
# ── Cell 0 — environment check: dataset resolution, GPU capability ──
# §3.1: both datasets must resolve before anything else (candidates
# first, rglob fallback in paths.py — a fallback resolution is fine,
# silence is not). §3.7: the GPU's COMPUTE CAPABILITY is checked, not
# just availability — torch reports available=True on a P100 whose
# sm_60 has no kernels in this build, and the crash then comes inside
# the first kernel launch, two hours into nothing.
import json
import sys
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

def find_dir(name: str) -> Path:
    hits = [p for p in KAGGLE_INPUT.rglob(name) if p.is_dir()]
    assert hits, (
        f"Dataset '{name}' is not attached. Use Add Input, then RESTART "
        f"the session — attaching does not mount into a running kernel."
    )
    assert len(hits) == 1, f"Ambiguous matches for '{name}': {hits}"
    return hits[0]

CODE_DIR = find_dir("dmrgt-phase-3-code")
sys.path.insert(0, str(CODE_DIR))

import paths  # noqa: E402  (resolves CACHE_DIR from dmrgt-phase-3-cache)
print("code dataset :", CODE_DIR)
print("cache dataset:", paths.CACHE_DIR, " (resolved by paths.py, §3.1)")
print("working dir  :", paths.OUTPUT_DIR)
print(json.dumps(paths.describe(), indent=2))

import verify_env  # noqa: E402

fp = verify_env.code_fingerprint()
print(f"[cell 0] code fingerprint: {fp}")
print("[cell 0]   ^ must equal the value printed by "
      "kaggle/build_code_dataset_phase3.sh (§1.3) — if it differs, the "
      "attached dmrgt-phase-3-code is a STALE version: remove + re-add "
      "it in Add Input, RESTART, compare again. Do not start training.")

problems = []
problems += verify_env.check_versions()   # versions + no-fill pct_change
problems += verify_env.check_gpu()        # name + sm_70+ + device count (§3.7)
problems += verify_env.check_bootstrap()  # the in-session Cell 6 path
assert not problems, (
    "environment check failed:\n  " + "\n  ".join(problems))
print("[cell 0] environment OK — one session: grid → winner → frozen "
      "graph → frozen arm → significance (§0).")

In [ ]:
# ── Cell 1 — cache integrity: all 11 parquets + hashes.json (§3.2) ──
# 4 base files + 7 flat grid variants, SHA-256 verified against
# hashes.json. `generated_at` is a timestamp key, never a file: the
# gate walks paths.CACHE_PARQUETS — an explicit 11-entry list (§3.2) —
# not hashes.json's keys, and graphs_frozen.parquet is deliberately
# not in that list (it is built in-session to /kaggle/working/, §4).
problems = verify_env.check_cache(require_variants=True)
assert not problems, "cache check failed:\n  " + "\n  ".join(problems)

# Import the whole in-session toolchain now — a missing or broken module
# must fail in minute one, not in hour three. (The staged-tree test
# exercises exactly this import set against a faithful copy locally.)
import data  # noqa: E402
import eval as ev  # noqa: E402,F401
import graph  # noqa: E402,F401
import sensitivity  # noqa: E402,F401
import splits  # noqa: E402,F401
import stats  # noqa: E402,F401
import train  # noqa: E402,F401

cfg = data.load_config(str(paths.CONFIG))
GRID_VARIANTS = list(sensitivity.variant_configs(cfg).keys())
assert GRID_VARIANTS == list(paths.GRID_VARIANT_NAMES), (
    f"variant disagreement: configs.yaml [graph_grid] derives "
    f"{GRID_VARIANTS} but paths.CACHE_PARQUETS pins "
    f"{list(paths.GRID_VARIANT_NAMES)}"
)
print(f"[cell 1] cache verified; {len(GRID_VARIANTS)} grid variants: "
      f"{GRID_VARIANTS}")

In [ ]:
# ── Cell 2 — stage 1: the 7-variant sensitivity grid (140 runs) ─────
# §1: dmrgt_all3 is trained ONCE per variant, each run emitting val AND
# test rows. The winner's already-existing test rows become the final
# dynamic result — dmrgt_all3 is NEVER retrained after this cell.
import time
import traceback

import torch

# Use EVERY T4 the accelerator provides: the §5 re-measurement
# (post-§3.5-fix) measured ~170 s/run on one T4, projecting ~11 h
# for the 160 runs — no margin under the session cap. Fold-level
# parallelism across both T4s is pure scheduling (each run reseeds
# itself), halving that to ~5.5–6 h.
DEVICES = ([f"cuda:{i}" for i in range(torch.cuda.device_count())]
           if torch.cuda.is_available() else ["cpu"])

N_RUNS_PER_VARIANT = 4 * len(cfg["training"]["seeds"])            # 20
N_RUNS_TOTAL = (len(GRID_VARIANTS) * N_RUNS_PER_VARIANT           # 140
                + N_RUNS_PER_VARIANT)                             # + 20 frozen

variant_seconds: dict = {}
failed: list = []
for name in GRID_VARIANTS:
    edge_file = paths.graph_variant_path(name)
    assert edge_file.exists(), (
        f"{edge_file} missing — attach the dmrgt-phase-3-cache version "
        f"carrying the FLAT graphs_<name>.parquet variants (cache root, "
        f"not a subdirectory: --dir-mode skip silently drops those) and "
        f"RESTART."
    )
    print("\n" + "=" * 70, flush=True)
    print(f"GRID VARIANT {name}  →  edges: {edge_file}", flush=True)
    print("=" * 70, flush=True)
    t0 = time.time()
    try:
        train.run_all(cfg, model_id="dmrgt_all3", edge_source=edge_file,
                      output_suffix=f"_grid_{name}",
                      devices=DEVICES)
    except Exception:
        # §3.6: ONE stream, flushed. print() writes to stdout and
        # traceback.print_exc writes to stderr; the two flush independently,
        # so a traceback can surface against the wrong variant in a
        # combined log. format_exc() into stdout keeps the order right.
        print(f"[grid] VARIANT {name} FAILED — moving on.", flush=True)
        print(traceback.format_exc(), flush=True)
        failed.append(name)
        continue
    dt = time.time() - t0
    variant_seconds[name] = dt
    per_run = dt / N_RUNS_PER_VARIANT
    done = len(variant_seconds)
    remaining_runs = ((len(GRID_VARIANTS) - done) * N_RUNS_PER_VARIANT
                      + N_RUNS_PER_VARIANT)
    projected = (sum(variant_seconds.values()) / done) * remaining_runs
    print(f"[grid] variant {name}: {dt / 60:.1f} min  "
          f"(~{per_run:.0f} s/run)", flush=True)
    print(f"[grid] projected remaining (grid tail + frozen arm): "
          f"{projected / 3600:.2f} h", flush=True)
    if name == GRID_VARIANTS[0]:
        # §5: re-derive the projection from THIS session's measurement —
        # never trust a number carried over from an earlier code state,
        # and re-measure after ANY change to the hot path.
        total_projected = per_run * N_RUNS_TOTAL
        print(f"[grid] §5 projection: {per_run:.0f} s/run × "
              f"{N_RUNS_TOTAL} runs ≈ {total_projected / 3600:.2f} h "
              f"(must clear the 12 h cap with margin)", flush=True)
        assert total_projected < 9 * 3600, (
            f"§5 TIMING HALT: measured {per_run:.0f} s/run projects "
            f"{total_projected / 3600:.2f} h for {N_RUNS_TOTAL} runs — "
            f"materially above the ~5.5-6 h the two-device design projects (§5 re-measured ~170 s/run single-GPU -> fold-parallel across both T4s). Stop and "
            f"re-derive the projection (§5) before committing; do not "
            f"burn the rest of the session hoping it speeds up."
        )

assert not failed, (
    f"{len(failed)} variant(s) FAILED: {failed}. Winner selection needs "
    f"all {len(GRID_VARIANTS)} grids complete — a partial grid must "
    f"never pick a winner from the variants that happened to finish. "
    f"The failed variants' tracebacks are in the log above."
)
print(f"\n[grid] all {len(GRID_VARIANTS)} variants completed — "
      f"{len(GRID_VARIANTS) * N_RUNS_PER_VARIANT} grid runs done.",
      flush=True)

In [ ]:
# ── Cell 3 — stage 2: winner selection, validation rows ONLY (§1) ───
# sensitivity.select_winner implements the spec's safeguard exactly:
# every grid prediction file is read once into `df`, asserted to hold
# only val/test splits, sliced to split == "val", and the full-frame
# reference is `del`eted before any scoring runs — so no code path can
# reach test rows through a lingering variable. The winner's existing
# test rows are first read in Cell 6, AFTER selection is locked
# (§1: stricter adherence to Rule 18, not looser).
record = sensitivity.select_winner(cfg)
winner = record["winner"]
winning_config = record["winning_graph_params"]

# §1: the required assertion, immediately after winner selection and
# before any test-split row is opened anywhere below.
assert set(winning_config) == {"k_pos", "k_neg", "corr_window"}, (
    f"winning_graph_params must be exactly the three graph knobs, got "
    f"{winning_config}"
)
assert winner in GRID_VARIANTS, f"winner {winner!r} not a grid variant"
assert (paths.OUTPUT_DIR / "phase3_frozen_graph_config.json").exists()
print(f"\n[cell 3] winner LOCKED: {winner} {winning_config}  "
      f"(val {record['selection_metric']}="
      f"{record['selection_value']:+.4f})", flush=True)

In [ ]:
# ── Cell 4 — stage 3: build graphs_frozen.parquet IN-SESSION (§2) ───
# Pure pandas/numpy over the ATTACHED prices.parquet — no dataset
# version bump, no re-attach, no restart (this cell is the one that
# used to require leaving Kaggle; it never needed to). The frozen graph
# is an OUTPUT written to /kaggle/working/ (paths.GRAPHS_FROZEN, §4),
# never part of the read-only cache. The winning k is merged into a
# COPY of the config — winning_graph_params only, never the whole
# selection record.
import copy

import pandas as pd

cfg_frozen = copy.deepcopy(cfg)
cfg_frozen["graph"].update(winning_config)
print(f"[cell 4] frozen graph at the winning k: {winning_config}")

prices = pd.read_parquet(paths.PRICES)
feats_dates = pd.read_parquet(paths.FEATURES, columns=["date"])
folds = splits.make_folds(cfg, available_dates=pd.DatetimeIndex(
    feats_dates["date"].unique()))

# Per-fold snapshots from each fold's train-end window only (Rule 16):
# build_frozen_graph_for_fold asserts the snapshot's correlation window
# ends at or before the fold's last training date, and
# build_all_frozen_graphs asserts identical edges within every fold and
# prints the per-date edge-count validation (640 = 320 +corr / 160
# -corr / 160 sector).
frozen_edges = graph.build_all_frozen_graphs(prices, cfg_frozen,
                                             folds=folds)
frozen_edges.to_parquet(paths.GRAPHS_FROZEN, index=False)
print(f"[cell 4] wrote {paths.GRAPHS_FROZEN}: {len(frozen_edges):,} rows, "
      f"{frozen_edges['fold'].nunique()} folds, "
      f"{frozen_edges.groupby(['fold', 'date']).ngroups} (fold, date) pairs")

In [ ]:
# ── Cell 5 — stage 4: train dmrgt_frozen at the winning k (20 runs) ──
# Same model class, folds, and seeds as the grid's dmrgt_all3 runs; the
# ONLY difference is the edge file (Rule 19) — the per-fold frozen
# graph just written to /kaggle/working/, read from
# paths.GRAPHS_FROZEN, never from the read-only cache (§4). Emits val
# AND test rows; Cell 6 scores the test rows exactly once.
train.run_all(cfg, model_id="dmrgt_frozen",
              edge_source=paths.GRAPHS_FROZEN,
              devices=DEVICES)
print("[cell 5] dmrgt_frozen trained — edges: graphs_frozen.parquet "
      "from /kaggle/working/ (in-session build, §2/§4)")

In [ ]:
# ── Cell 6 — stage 5: significance tests, in-session (§6) ───────────
# The FIRST read of any test-split row anywhere in this session: the
# winner is locked (Cell 3), so its already-existing grid test rows
# become the final dynamic-arm result (§1 — never retrained, never
# re-selected). Both arms flow through the same evaluation pipeline
# with the declared-in-advance identity post-processing (Rule 19).
import json

import pandas as pd

dynamic_pred = pd.read_parquet(
    paths.OUTPUT_DIR / f"predictions_dmrgt_all3_grid_{winner}.parquet")
dynamic_test = dynamic_pred[dynamic_pred["split"] == "test"]
frozen_pred = pd.read_parquet(
    paths.OUTPUT_DIR / "predictions_dmrgt_frozen.parquet")
frozen_test = frozen_pred[frozen_pred["split"] == "test"]

wilcoxon_result = stats.wilcoxon_test(
    dynamic_ic=stats.per_fold_seed_ic(dynamic_test, cfg),
    frozen_ic=stats.per_fold_seed_ic(frozen_test, cfg),
    cfg=cfg,
)
bootstrap_result = stats.bootstrap_sharpe_test(
    dynamic_returns=stats.daily_returns(dynamic_test, cfg),
    frozen_returns=stats.daily_returns(frozen_test, cfg),
    cfg=cfg,
)

sig_out = {
    "wilcoxon": wilcoxon_result,
    "bootstrap_sharpe": bootstrap_result,
    "winning_graph_config": winning_config,
    "conclusion": (
        f"Dynamic graph rank IC exceeds frozen in {wilcoxon_result['wins']}/"
        f"{wilcoxon_result['n_pairs']} folds "
        f"(p={wilcoxon_result['p_value']:.3f}); "
        f"Sharpe difference p={bootstrap_result['bootstrap_p_value']:.3f}"
    ),
}
sig_path = paths.OUTPUT_DIR / "phase3_significance_results.json"
sig_path.write_text(json.dumps(sig_out, indent=2))

wil, boot = wilcoxon_result, bootstrap_result
print(f"[cell 6] Wilcoxon (paired rank IC): wins {wil['wins']}/"
      f"{wil['n_pairs']}  statistic={wil['statistic']:.4f}  "
      f"p={wil['p_value']:.4f}  significant={wil['significant']}")
print(f"[cell 6]   dynamic IC {wil['dynamic_mean_rank_ic']:+.5f} vs "
      f"frozen {wil['frozen_mean_rank_ic']:+.5f}  (paired diff "
      f"{wil['mean_paired_difference']:+.5f})")
print(f"[cell 6] Stationary bootstrap (Sharpe difference): "
      f"dyn {boot['observed_sharpe_dynamic']:+.3f} vs "
      f"fro {boot['observed_sharpe_frozen']:+.3f} "
      f"(diff {boot['observed_sharpe_difference']:+.3f}, 95% CI "
      f"[{boot['ci_95'][0]:+.3f}, {boot['ci_95'][1]:+.3f}])")
print(f"[cell 6]   one-sided p={boot['bootstrap_p_value']:.4f}  "
      f"two-sided p={boot['p_two_sided']:.4f}")
print(f"[cell 6] CONCLUSION: {sig_out['conclusion']}")
print(f"[cell 6] wrote {sig_path}")

In [ ]:
# ── Cell 7 — output verification: nothing ends until this passes (§7) ─
# A committed session's Output tab only preserves what actually landed
# in /kaggle/working/ — confirm every required file BEFORE closing.
required = [
    "phase3_sensitivity_grid.csv",
    "phase3_frozen_graph_config.json",
    "graphs_frozen.parquet",
    "predictions_dmrgt_frozen.parquet",
    "phase3_significance_results.json",
    "run_manifest.json",
] + [f"predictions_dmrgt_all3_grid_{v}.parquet" for v in GRID_VARIANTS]
missing = [f for f in required
           if not (paths.OUTPUT_DIR / f).exists()]
assert not missing, f"Missing outputs before session end: {missing}"
print(f"[verify] all {len(required)} expected files present")

# Row-count audit (§3.4): per-fold counts SUMMED — the four folds' test
# windows differ (252/253/252/251 trading days), so multiplying any one
# fold's count is the false-mismatch trap. Each variant's expected count
# is independent — never summed across variants.
n_seeds = len(cfg["training"]["seeds"])
n_stocks = len(cfg["universe"]["tickers"])
val_days = sum(len(f.val_dates) for f in folds)
test_days = sum(len(f.test_dates) for f in folds)
exp_val = val_days * n_seeds * n_stocks
exp_test = test_days * n_seeds * n_stocks
for name in GRID_VARIANTS:
    df = pd.read_parquet(paths.OUTPUT_DIR
                         / f"predictions_dmrgt_all3_grid_{name}.parquet")
    got = df["split"].value_counts().to_dict()
    assert got.get("val", 0) == exp_val and got.get("test", 0) == exp_test, (
        f"{name}: val {got.get('val', 0):,} (expected {exp_val:,}), "
        f"test {got.get('test', 0):,} (expected {exp_test:,})"
    )
frozen_rows = pd.read_parquet(
    paths.OUTPUT_DIR / "predictions_dmrgt_frozen.parquet")
assert len(frozen_rows) == (val_days + test_days) * n_seeds * n_stocks, (
    f"dmrgt_frozen rows {len(frozen_rows):,} != "
    f"{(val_days + test_days) * n_seeds * n_stocks:,}"
)
print(f"[verify] row counts: 7 grid files at {exp_val:,} val + "
      f"{exp_test:,} test rows each; dmrgt_frozen at "
      f"{len(frozen_rows):,} rows ✓")

print("\nNEXT STEPS (local, after this commit finishes — §8):")
print("1. Download EVERYTHING in /kaggle/working/ → move into artifacts/")
print("2. python compare.py   # one-shot report; cross-checks Cell 6's "
      "p-values against phase3_significance_results.json")
print("3. git commit; git tag -a phase3-complete")
print("\n*Research project for academic purposes only — not investment "
      "advice.*")